In [2]:
import tensorflow as tf
import numpy as np
import pandas as pd
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import accuracy_score, log_loss
from tensorflow.keras.callbacks import EarlyStopping
import os
os.chdir("D:\Machine Learning\Datasets\Datasets\CatsVsDogs")

In [4]:
# https://www.kaggle.com/datasets/marquis03/cats-and-dogs
train_dir = "D:\Machine Learning\Datasets\Datasets\CatsVsDogs\\train"
test_dir = "D:\Machine Learning\Datasets\Datasets\CatsVsDogs\\val"
train_datagen = ImageDataGenerator(rescale =1.0/255.)
test_datagen = ImageDataGenerator(rescale =1.0/255.)
train_generator = train_datagen.flow_from_directory(train_dir,
                    target_size=(150,150), batch_size= 32,class_mode='binary')
test_generator = test_datagen.flow_from_directory(test_dir,
                    target_size=(150,150), batch_size= 32,class_mode='binary')

Found 275 images belonging to 2 classes.
Found 70 images belonging to 2 classes.


In [20]:
tf.random.set_seed(25)
model = tf.keras.Sequential([
    tf.keras.layers.InputLayer(input_shape=(150,150,3)), # 3-> colourd image (red,blue, green) and look for X_test fror size
    tf.keras.layers.Conv2D(16,kernel_size=(3,3),activation='relu'),# filter = 16
    tf.keras.layers.MaxPooling2D(pool_size=(2,2)),
    tf.keras.layers.Conv2D(32,kernel_size=(3,3),activation='relu'), # filter = 32
    tf.keras.layers.MaxPooling2D(pool_size=(2,2)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(256, activation='relu'), # fully connected layer
    tf.keras.layers.Dense(1, activation='sigmoid')]) # fully connected layer

print(model.summary())


Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_10 (Conv2D)              │ (None, 148, 148, 16)   │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_10 (MaxPooling2D) │ (None, 74, 74, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_11 (Conv2D)              │ (None, 72, 72, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_11 (MaxPooling2D) │ (None, 36, 36, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_5 (Flatten)             │ (None, 41472)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 41472)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 256)            │    10,617,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,622,433 (40.52 MB)

 Trainable params: 10,622,433 (40.52 MB)

 Non-trainable params: 0 (0.00 B)

None


In [21]:
# ImageDataGenerator will randomly take some photos from that particular folder 32 one at a time (i.e. images are taken in batches)
monitor = EarlyStopping(monitor='val_loss',min_delta=1e-3, patience=5,
                        verbose=1, restore_best_weights = True)
model.compile(optimizer='Adam',loss='binary_crossentropy',metrics=['acc'])
history = model.fit(train_generator,validation_data=test_generator,epochs=100,verbose=2,callbacks=[monitor])
log_loss, acc = model.evaluate(test_generator)
print(log_loss, acc)

Epoch 1/100
9/9 - 4s - 446ms/step - acc: 0.5891 - loss: 2.5047 - val_acc: 0.6571 - val_loss: 1.2421
Epoch 2/100
9/9 - 3s - 283ms/step - acc: 0.6545 - loss: 0.7566 - val_acc: 0.6571 - val_loss: 0.6535
Epoch 3/100
9/9 - 3s - 289ms/step - acc: 0.6764 - loss: 0.5721 - val_acc: 0.6714 - val_loss: 0.6585
Epoch 4/100
9/9 - 3s - 281ms/step - acc: 0.7200 - loss: 0.4870 - val_acc: 0.6286 - val_loss: 0.7140
Epoch 5/100
9/9 - 3s - 285ms/step - acc: 0.8436 - loss: 0.3917 - val_acc: 0.4857 - val_loss: 0.7463
Epoch 6/100
9/9 - 3s - 283ms/step - acc: 0.9236 - loss: 0.2897 - val_acc: 0.6429 - val_loss: 0.8270
Epoch 7/100
9/9 - 2s - 276ms/step - acc: 0.9345 - loss: 0.2099 - val_acc: 0.5429 - val_loss: 0.8782
Epoch 7: early stopping
Restoring model weights from the end of the best epoch: 2.
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step - acc: 0.6571 - loss: 0.6535
0.653465747833252 0.6571428775787354
